# TinyCD · LEVIR-CD pilot ve final

Notebook eğitim kodunu GitHub reposundan alır ve kullanılan commit'i gösterir.
Varsayılan **review** modu yalnız kaynakları doğrular. Pilot için ayrıca eğitim onayı verildiğinde aşağıdaki adımları izle.

1. Colab'da yeni bir **T4 GPU** oturumu seç.
2. Varsayılan DATA_SOURCE="auto" ile notebook eski Colab akışındaki train ve validation
   arşivlerini gdown ile Colab'ın yerel diskine indirir. Geçerli yerel veri yeniden indirilmez.
   Train ve validation için sırasıyla 445 ve 64 eksiksiz A/B/label çifti doğrulanır.
   Hazır Drive ZIP'leri için DATA_SOURCE="zip", açılmış Drive klasörü için "folder" seçilebilir.
3. Eğitim onayından sonra MODE değerini **pilot** yap, RUN_ID için benzersiz bir
   ad belirle ve hücreleri sırayla çalıştır. İlk adım batch 8 T4 kontrolüdür.
   Ardından pilot temiz başlangıçla çalışır. Epoch 9 sonunda notebook aynı
   koşuyu yeni Python sürecinde açıp epoch 10'un ilk üç batch'ini doğrular.
4. Başarılı koşuda Drive'daki **pilot_summary.json**, **pilot_curves.png** ve
   **resume_probe.json** dosyalarını paylaş. Kesinti olursa aynı MODE ve RUN_ID ile
   tekrar çalıştır. GitHub ana dalı ilerlediyse PROJECT_REF alanına ilk koşunun commit'ini yaz. Final için **continue**, pilot değerlendirmesi ve devam onayından
   sonra aynı RUN_ID ile kullanılır.

MODE=**audit** yalnız train/val veri denetimi yapar. Standart Colab Drive bağlantısı
MyDrive dizinini Dosyalar panelinde gösterebilir. Notebook Drive kökünü listelemez.
Çıktılar doğrudan change-detection-runs/stage1/RUN_ID/tinycd-training altına yazılır.


In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path, PurePosixPath
import os
import re
import shutil
import subprocess
import sys
from tempfile import TemporaryDirectory
from uuid import uuid4
from zipfile import BadZipFile, ZipFile

MODE = "review"  # review | audit | pilot | continue
PROJECT_REF = "main"  # Devamda gerekirse ilk koşunun Git commit SHA’sını gir.
RUN_ID = ""      # Örnek: tinycd-pilot-01. Devamda aynı kimliği kullan.

DATA_SOURCE = "auto"  # auto | zip | folder
# Yalnız alternatif kipler için.
DATA_SOURCE_ROOT = "/content/drive/MyDrive/LEVIR-CD"  # folder
TRAIN_ZIP = "/content/drive/MyDrive/levir-train.zip"  # zip
VAL_ZIP = "/content/drive/MyDrive/levir-val.zip"      # zip

WORK = Path("/content/change-detection-stage1/tinycd-training")
DRIVE_RUNS = Path("/content/drive/MyDrive/change-detection-runs/stage1")
REPO_URL = "https://github.com/apeates/change-detection.git"
ENV_PY = WORK / "venv/bin/python"

if not re.fullmatch(r"[a-zA-Z0-9][a-zA-Z0-9_./-]{0,127}", PROJECT_REF):
    raise ValueError("Geçerli bir Git ref veya commit SHA gir.")
if DATA_SOURCE not in {"auto", "zip", "folder"}:
    raise ValueError("DATA_SOURCE auto, zip veya folder olmalı.")
if MODE not in {"review", "audit", "pilot", "continue"}:
    raise ValueError("MODE review, audit, pilot veya continue olmalı.")
if MODE != "review" and not re.fullmatch(r"[a-zA-Z0-9][a-zA-Z0-9_-]{0,79}", RUN_ID):
    raise ValueError("RUN_ID ver. Yalnız harf, rakam, tire ve alt çizgi kullan.")
print(f"Mod: {MODE}. " + ("Eğitim komutu çalışmayacak." if MODE == "review" else f"Koşu: {RUN_ID}"))


In [ ]:
SPLIT_COUNTS = {"train": 445, "val": 64}
DRIVE_FILE_IDS = {
    "train": "18GuoCuBn48oZKAlEo-LrNwABrFhVALU-",
    "val": "1BqSt4ueO7XAyQ_84mUjswUSJt13ZBuzG",
}
SIDES = ("A", "B", "label")
PROJECT_FILES = ('train_tinycd.py', 'tinycd_data.py', 'levir_data.py', 'levir_tiles.py', 'baseline_diff.py', 'requirements-tinycd.txt', 'requirements-train.txt', 'requirements.txt')


def checkout_project(directory):
    subprocess.run(
        ["git", "clone", "--quiet", "--depth=1", "--filter=blob:none",
         "--no-checkout", "--branch", "main", REPO_URL, str(directory)],
        check=True,
    )
    # Materialize only the scripts and requirements used by this notebook.
    subprocess.run(["git", "sparse-checkout", "set", "--no-cone", "--",
                    *(f"/{name}" for name in PROJECT_FILES)], cwd=directory, check=True)
    ref = "origin/main"
    if PROJECT_REF != "main":
        subprocess.run(["git", "fetch", "--quiet", "--depth=1", "origin", PROJECT_REF],
                       cwd=directory, check=True)
        ref = "FETCH_HEAD"
    subprocess.run(["git", "checkout", "--quiet", "--detach", ref], cwd=directory, check=True)
    revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=directory, text=True).strip()
    for name in PROJECT_FILES:
        path = directory / name
        if not path.is_file():
            raise RuntimeError(f"{revision} commit'inde {name} bulunamadı.")
        if name.endswith(".py"):
            compile(path.read_bytes(), name, "exec")
    print(f"GitHub commit: {revision}")
    return revision


def install_runtime():
    if sys.version_info < (3, 11):
        raise RuntimeError("Python 3.11 veya üzeri gerekli.")
    if not ENV_PY.is_file():
        subprocess.run([sys.executable, "-m", "venv", "--system-site-packages",
                        str(ENV_PY.parent.parent)], check=True)
    pins = {"numpy": "2.4.6", "opencv-python-headless": "4.10.0.84"}
    check = ("from importlib.metadata import version; "
             f"assert all(version(package) == expected for package, expected in {pins!r}.items()); "
             "import cv2, numpy; assert cv2.__version__ == '4.10.0'")
    if subprocess.run([str(ENV_PY), "-c", check], capture_output=True).returncode:
        subprocess.run([str(ENV_PY), "-m", "pip", "install", "--ignore-installed", "--no-deps",
                        *(f"{pkg}=={ver}" for pkg, ver in pins.items())], check=True)
    subprocess.run([str(ENV_PY), "-c", check + "; import torch, torchvision, PIL, matplotlib"], check=True)


def expected_files(split):
    return {(side, f"{split}_{i}.png")
            for side in SIDES for i in range(1, SPLIT_COUNTS[split] + 1)}


def verify_layout(directory, split):
    actual = set()
    for side in SIDES:
        folder = directory / side
        if not folder.is_dir():
            raise ValueError(f"Eksik klasör: {folder}")
        for path in folder.glob("*.png"):
            if not path.is_file() or path.stat().st_size == 0:
                raise ValueError(f"Eksik veya boş PNG: {path}")
            actual.add((side, path.name))
    if actual != expected_files(split):
        raise ValueError(f"{split}: eksik veya beklenmeyen PNG var; {SPLIT_COUNTS[split]} çift bekleniyor.")


def archive_entries(archive, split):
    expected = expected_files(split)
    found = {}
    for info in archive.infolist():
        member = PurePosixPath(info.filename)
        if info.is_dir() or len(member.parts) < 2:
            continue
        entry = (member.parent.name, member.name)
        if entry not in expected:
            continue
        if member.is_absolute() or ".." in member.parts or any(
            part in {"train", "val", "test"} and part != split for part in member.parts[:-2]
        ):
            raise ValueError(f"Arşivde beklenmeyen yol: {member}")
        if entry in found:
            raise ValueError(f"Arşivde yinelenen girdi: {entry}")
        if not 0 < info.file_size <= 16 * 1024**2:
            raise ValueError(f"Beklenmeyen PNG boyutu: {member}")
        found[entry] = info
    if found.keys() != expected:
        raise ValueError(f"{split}: {len(expected) - len(found)} beklenen PNG eksik.")
    return found


def copy_split(split, target, *, archive_path=None, folder_source=None):
    # Publish only after all expected members have been completely read and verified.
    with TemporaryDirectory(dir=target.parent) as temporary:
        staged = Path(temporary) / split
        for side in SIDES:
            (staged / side).mkdir(parents=True)
        if folder_source is not None:
            verify_layout(folder_source, split)
            for side, name in sorted(expected_files(split)):
                shutil.copyfile(folder_source / side / name, staged / side / name)
        else:
            with ZipFile(archive_path) as archive:
                for (side, name), info in archive_entries(archive, split).items():
                    with archive.open(info) as stream, (staged / side / name).open("xb") as output:
                        shutil.copyfileobj(stream, output)
        verify_layout(staged, split)
        staged.replace(target)


def download_archive(split, path):
    check = "from importlib.metadata import version; assert version('gdown') == '5.2.0'"
    python = str(ENV_PY)
    if subprocess.run([python, "-c", check], capture_output=True).returncode:
        subprocess.run([python, "-m", "pip", "install", "gdown==5.2.0"], check=True)
    code = ("import gdown, sys; "
            "result = gdown.download(id=sys.argv[1], output=sys.argv[2], quiet=False); "
            "raise SystemExit(0 if result else 1)")
    subprocess.run([python, "-c", code, DRIVE_FILE_IDS[split], str(path)], check=True)


def prepare_auto_split(split, target):
    archive_cache = WORK / "archives"
    archive_cache.mkdir(parents=True, exist_ok=True)
    cached = archive_cache / f"{split}-{DRIVE_FILE_IDS[split]}.zip"
    if cached.is_file():
        try:
            copy_split(split, target, archive_path=cached)
        except (BadZipFile, ValueError, EOFError) as error:
            print(f"Geçersiz yerel {split} arşivi yeniden indirilecek: {error}")
            cached.unlink()
        else:
            print(f"Yerel {split} arşivi yeniden kullanıldı.")
            return
    # gdown writes into a private staging directory. An interrupted run leaves
    # no final cache file for the next run to treat as valid.
    with TemporaryDirectory(prefix=f"{split}-", dir=archive_cache) as temporary:
        pending = Path(temporary) / f"{split}.zip"
        download_archive(split, pending)
        copy_split(split, target, archive_path=pending)
        pending.replace(cached)


def local_train_val():
    archives = {"train": Path(TRAIN_ZIP), "val": Path(VAL_ZIP)}
    if DATA_SOURCE == "auto":
        identity = ["auto", DRIVE_FILE_IDS]
    elif DATA_SOURCE == "folder":
        identity = ["folder", str(Path(DATA_SOURCE_ROOT))]
    else:
        identity = ["zip", [[str(p), p.stat().st_size, p.stat().st_mtime_ns] for p in archives.values()]]
    inputs_id = sha256(repr(identity).encode()).hexdigest()[:16]
    cache = WORK / "inputs" / inputs_id
    cache.mkdir(parents=True, exist_ok=True)
    for split, count in SPLIT_COUNTS.items():
        target = cache / split
        if target.exists():
            try:
                verify_layout(target, split)
            except ValueError:
                print(f"Eksik yerel {split} klasörü yeniden hazırlanacak.")
                shutil.rmtree(target)
        if not target.exists():
            if DATA_SOURCE == "auto":
                prepare_auto_split(split, target)
            elif DATA_SOURCE == "folder":
                copy_split(split, target, folder_source=Path(DATA_SOURCE_ROOT) / split)
            else:
                copy_split(split, target, archive_path=archives[split])
        print(f"{split}: {count} A/B/label çifti doğrulandı.")
    return cache


def run_process(command, log_path, project):
    with log_path.open("x", encoding="utf-8", buffering=1) as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, cwd=project,
                              env={**os.environ, "MPLBACKEND": "Agg",
                                   "MPLCONFIGDIR": str(WORK / "mpl")}) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                return process.wait()
            except KeyboardInterrupt:
                process.terminate()
                raise


def main():
    WORK.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(prefix="source-", dir=WORK) as workspace:
        project = Path(workspace) / "repo"
        revision = checkout_project(project)
        if MODE == "review":
            print("GitHub kaynakları doğrulandı. Review modu tamamlandı.")
            return
        run_selected_mode(project, revision)


def run_selected_mode(project, revision):
    if MODE in {"pilot", "continue"}:
        gpu = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).splitlines()
        if len(gpu) != 1 or gpu[0].strip() not in {"Tesla T4", "NVIDIA T4", "T4"}:
            raise RuntimeError(f"Tesla T4 gerekli. Colab GPU listesi: {gpu}")
    from google.colab import drive
    drive.mount("/content/drive")
    output = DRIVE_RUNS / RUN_ID / "tinycd-training"
    print(f"Drive çıktı klasörü: {output}")
    install_runtime()
    dataset = local_train_val()
    log_dir = output / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    command = [
        str(ENV_PY), "-u", str(project / "train_tinycd.py"), "--mode", MODE,
        "--revision", revision,
        "--data-root", str(dataset), "--cache-dir", str(WORK / "cache" / RUN_ID),
        "--output-dir", str(output),
    ]
    for attempt in (1, 2):
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        log_path = log_dir / f"{MODE}-{stamp}-{uuid4().hex[:8]}.log"
        code = run_process(command, log_path, project)
        if code == 0:
            return
        if code != 75 or MODE != "pilot" or attempt == 2:
            raise RuntimeError(f"Koşu hata kodu {code}. Günlük: {log_path}")
        print("Epoch 9 checkpoint'i hazır. Epoch 10 için yeni süreç açılıyor.")


main()
